# Music Annotation Project V7.9.1: Safety-First Challenger Confirmation and Resumable Stage 4

**Runtime:** CPU. Select **Runtime > Run all**.

This continuation does not repeat coarse or fine discovery. It:

1. verifies the completed V7.8 baseline-equivalence table;
2. re-analyses the saved coarse, fine and all-page checkpoints;
3. uses three behaviourally distinct safety-first challenger families;
4. reuses existing all-page rows and computes only missing candidate/page rows;
5. runs Stage 4 page-atomically with durable completion markers;
6. resumes after interruption and generates diagnostics only after all pages finish.

The incumbent remains `geometry_t80`. Challenger results are evidence only and do not authorise promotion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2, json, hashlib, gc, re, math, shutil, ast
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
# Point this to the crashed Notebook 22 run containing tables/ and finalists/.
SOURCE_RUN=None
# Point this to the original Notebook 22 source. The continuation imports its tested engine definitions without executing its long stages.
ORIGINAL_NOTEBOOK=ROOT/'notebooks'/'22_V7_9_Observed_Ink_Glyph_Preservation_Challenger.ipynb'
OUTPUT_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_9_1'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')+'_safety_first_resumable'
OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables'; PAGE_RESULTS=OUT/'stage4_pages'; FINALISTS=OUT/'finalists'; CONTACTS=OUT/'contact_sheets'; AUDIT=OUT/'audit'
for p in [OUT,TABLES,PAGE_RESULTS,FINALISTS,CONTACTS,AUDIT]: p.mkdir(parents=True,exist_ok=True)
FORCE_CONFIRMATION=False
FORCE_STAGE4=False
MAX_PAGES=None

def latest_complete(root):
    hits=[]
    if root.exists():
        for p in root.rglob('selected_finalists.csv'):
            r=p.parent.parent
            if (r/'tables'/'fine_discovery_route_metrics_checkpoint.csv').exists(): hits.append(r)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
if SOURCE_RUN is None: SOURCE_RUN=latest_complete(ROOT)
if SOURCE_RUN is None: raise RuntimeError('Set SOURCE_RUN to the crashed Notebook 22 run.')
SOURCE_RUN=Path(SOURCE_RUN)
print('Source run:',SOURCE_RUN)
print('Output:',OUT)

## Safety-first candidate decision

The original shortlist over-weighted absolute false-preservation count and admitted duplicate consensus widths. This continuation uses three distinct route behaviours:

- `connected_dark`: strongest fine-discovery safety signal;
- `union`: a materially different, higher-coverage route family;
- `consensus`: conservative continuity with the already confirmed candidate.

All three must receive all-page confirmation before Stage 4. The code fails closed if a candidate has no positive true-print gain, no gain pages, or non-finite metrics.

In [ ]:
# Predeclared, behaviourally distinct candidates. Two require missing all-page confirmation.
SELECTED_CANDIDATES=[
 'v79_connected_dark_d80_w1.75_a12_100',
 'v79_union_d100_w4.0_a12_1800',
 'v79_consensus_d100_w1.75_a18_100',
]
SELECTION_RATIONALE={
 SELECTED_CANDIDATES[0]:'connected_dark family; best displayed fine-discovery global precision/utility family',
 SELECTED_CANDIDATES[1]:'union family; distinct higher-coverage behaviour and better coarse utility than shortlisted consensus',
 SELECTED_CANDIDATES[2]:'consensus family; lowest false-preservation count among existing all-page confirmed candidates',
}
pd.DataFrame([{'candidate':c,'rank':i+1,'rationale':SELECTION_RATIONALE[c]} for i,c in enumerate(SELECTED_CANDIDATES)]).to_csv(TABLES/'selected_finalists_safety_first.csv',index=False)
display(pd.read_csv(TABLES/'selected_finalists_safety_first.csv'))

In [ ]:
# Integrity and checkpoint loading.
def req(name):
 p=SOURCE_RUN/'tables'/name
 if not p.exists(): raise RuntimeError(f'Missing required checkpoint: {p}')
 return p
baseline=pd.read_csv(req('baseline_equivalence.csv'))
if not (baseline.geometry_exact.astype(bool).all() and (baseline.different_pixels==0).all() and baseline.outside_immutable.astype(bool).all()):
 raise RuntimeError('Frozen V7.8 baseline equivalence did not pass exactly.')
coarse=pd.read_csv(req('coarse_route_summary.csv'))
fine=pd.read_csv(req('fine_route_summary.csv'))
confirmation=pd.read_csv(req('all_page_confirmation_route_metrics_checkpoint.csv'))
roles=pd.read_csv(req('development_page_roles.csv'))
pages=roles.page.astype(str).tolist()
if MAX_PAGES is not None: pages=pages[:MAX_PAGES]
print('Baseline pages:',len(baseline),'Confirmation rows:',len(confirmation),'Pages:',len(pages))

## Import the original tested Notebook 22 engine without running Stages 1–4

This extracts imports, function definitions and simple configuration assignments from the original notebook. It deliberately excludes loops and expression statements, so the expensive original stages are not executed. The continuation then checks that the required candidate evaluator and page loader exist.

In [ ]:
if not ORIGINAL_NOTEBOOK.exists(): raise RuntimeError(f'Original Notebook 22 not found: {ORIGINAL_NOTEBOOK}')
raw=json.loads(ORIGINAL_NOTEBOOK.read_text())
engine_nodes=[]
for cell in raw.get('cells',[]):
 if cell.get('cell_type')!='code': continue
 src=''.join(cell.get('source',[]))
 src='\n'.join(line for line in src.splitlines() if not line.lstrip().startswith(('%','!')))
 try: tree=ast.parse(src)
 except SyntaxError: continue
 for node in tree.body:
  if isinstance(node,(ast.Import,ast.ImportFrom,ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef)):
   engine_nodes.append(node)
  elif isinstance(node,ast.Assign):
   # Retain only assignments that do not call processing functions.
   if not any(isinstance(x,(ast.Call,ast.ListComp,ast.DictComp,ast.SetComp,ast.GeneratorExp)) for x in ast.walk(node.value)):
    engine_nodes.append(node)
module=ast.Module(body=engine_nodes,type_ignores=[]);ast.fix_missing_locations(module)
ENGINE={'ROOT':ROOT,'SOURCE_RUN':SOURCE_RUN}
exec(compile(module,str(ORIGINAL_NOTEBOOK),'exec'),ENGINE)
print('Imported engine functions:',sorted(k for k,v in ENGINE.items() if callable(v))[:80])

def resolve_callable(names):
 for n in names:
  if callable(ENGINE.get(n)): return ENGINE[n],n
 raise RuntimeError('Original Notebook 22 engine is missing required callable; tried '+', '.join(names))
load_page,LOAD_NAME=resolve_callable(['load_page','page_invariants','load_page_invariants','prepare_page'])
evaluate_candidate,EVAL_NAME=resolve_callable(['evaluate_candidate','evaluate_route_candidate','route_metrics_for_candidate','evaluate_route'])
build_candidate,BUILD_NAME=resolve_callable(['build_candidate','build_route_candidate','candidate_reconstruction','reconstruct_candidate'])
print('Resolved:',LOAD_NAME,EVAL_NAME,BUILD_NAME)

In [ ]:
# Adapter calls tolerate the common signatures used across Notebook 22 revisions.
import inspect

def call_flexible(fn, **kwargs):
 sig=inspect.signature(fn)
 accepted={k:v for k,v in kwargs.items() if k in sig.parameters}
 missing=[n for n,p in sig.parameters.items() if p.default is inspect._empty and p.kind in (p.POSITIONAL_ONLY,p.POSITIONAL_OR_KEYWORD) and n not in accepted]
 if missing: raise RuntimeError(f'{fn.__name__} adapter missing required arguments: {missing}')
 return fn(**accepted)

def get_page(page):
 return call_flexible(load_page,page=page,page_name=page,name=page)

def metric_row(page,candidate,page_data):
 result=call_flexible(evaluate_candidate,page=page,page_name=page,candidate=candidate,candidate_name=candidate,data=page_data,inv=page_data,page_data=page_data)
 if isinstance(result,pd.Series): result=result.to_dict()
 if not isinstance(result,dict): raise RuntimeError(f'{candidate} {page}: evaluator must return dict/Series')
 return {'stage':'all_page_confirmation','page':page,'candidate':candidate,**result}

def candidate_arrays(page,candidate,page_data):
 result=call_flexible(build_candidate,page=page,page_name=page,candidate=candidate,candidate_name=candidate,data=page_data,inv=page_data,page_data=page_data)
 if isinstance(result,np.ndarray): return {'reconstruction':result}
 if isinstance(result,dict): return result
 raise RuntimeError(f'{candidate} {page}: builder must return ndarray or dict')

## Incremental all-page confirmation

Existing rows are preserved. Only missing candidate/page pairs are calculated. The checkpoint is written atomically after every page.

In [ ]:
CONFIRM_PATH=TABLES/'all_page_confirmation_route_metrics_checkpoint.csv'
base=confirmation.copy()
if CONFIRM_PATH.exists() and not FORCE_CONFIRMATION:
 base=pd.concat([base,pd.read_csv(CONFIRM_PATH)],ignore_index=True).drop_duplicates(['page','candidate'],keep='last')

def atomic_csv(df,path):
 tmp=Path(str(path)+'.tmp');df.to_csv(tmp,index=False);tmp.replace(path)

for pi,page in enumerate(pages,1):
 missing=[c for c in SELECTED_CANDIDATES if FORCE_CONFIRMATION or not ((base.page.astype(str)==page)&(base.candidate.astype(str)==c)).any()]
 if not missing: continue
 data=get_page(page);new=[]
 for candidate in missing:
  row=metric_row(page,candidate,data);new.append(row);print('Confirmed',page,candidate)
 base=pd.concat([base,pd.DataFrame(new)],ignore_index=True).drop_duplicates(['page','candidate'],keep='last')
 atomic_csv(base,CONFIRM_PATH)
 del data;gc.collect()

selected_rows=base[base.candidate.isin(SELECTED_CANDIDATES)].copy()
counts=selected_rows.groupby('candidate').page.nunique()
if any(counts.get(c,0)!=len(pages) for c in SELECTED_CANDIDATES): raise RuntimeError(f'Incomplete all-page confirmation: {counts.to_dict()}')
summary=selected_rows.groupby('candidate',as_index=False).agg(pages=('page','nunique'),added_true_print=('added_true_print','sum'),added_false_annotation=('added_false_annotation','sum'),mean_precision=('added_precision','mean'),median_precision=('added_precision','median'),mean_recall_gain=('clean_print_recall_gain','mean'),pages_with_gain=('added_true_print',lambda x:int((x>0).sum())),pages_with_false_preservation=('added_false_annotation',lambda x:int((x>0).sum())))
summary['global_precision']=summary.added_true_print/(summary.added_true_print+summary.added_false_annotation).clip(lower=1)
summary['false_per_true']=summary.added_false_annotation/summary.added_true_print.clip(lower=1)
summary.to_csv(TABLES/'all_page_route_summary_safety_first.csv',index=False)
if (summary.added_true_print<=0).any() or (summary.pages_with_gain<=0).any(): raise RuntimeError('At least one challenger produced no all-page true-print gain.')
display(summary.sort_values(['pages_with_false_preservation','global_precision'],ascending=[True,False]))

## Resumable page-atomic Stage 4

Each page writes reconstruction arrays, page metrics, component metrics and a completion marker. A marker is accepted only when its candidate list and output hashes still match. No full-page Boolean component masks are retained. Component rows are flushed per page, and contact sheets are deferred.

In [ ]:
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()

def valid_complete(page):
 marker=PAGE_RESULTS/f'{page}_complete.json'
 if FORCE_STAGE4 or not marker.exists(): return False
 try:
  m=json.loads(marker.read_text())
  if m.get('candidates')!=SELECTED_CANDIDATES:return False
  return all((OUT/rel).exists() and sha256_file(OUT/rel)==digest for rel,digest in m.get('outputs',{}).items())
 except Exception:return False

def extract_reconstruction(arrays):
 for key in ['reconstruction','recon','image','result','candidate']:
  if key in arrays and isinstance(arrays[key],np.ndarray):return arrays[key]
 vals=[v for v in arrays.values() if isinstance(v,np.ndarray) and v.ndim==2]
 if len(vals)==1:return vals[0]
 raise RuntimeError('Could not identify reconstruction array.')

for index,page in enumerate(pages,1):
 if valid_complete(page): print('Resume skip',page);continue
 data=get_page(page)
 page_metric_rows=[];component_rows=[];output_hashes={}
 # Construct and save only the three selected candidates.
 for candidate in SELECTED_CANDIDATES:
  arrays=candidate_arrays(page,candidate,data);recon=extract_reconstruction(arrays)
  # Fail closed on outside-mask drift whenever observed/mask are exposed by the engine.
  observed=(data.get('observed') if isinstance(data,dict) else None);mask=(data.get('mask') if isinstance(data,dict) else None)
  if observed is not None and mask is not None and not np.array_equal(recon[mask==0],observed[mask==0]):raise RuntimeError(f'{page} {candidate}: outside-mask drift')
  npz=FINALISTS/f'{page}_{candidate}.npz';np.savez_compressed(npz,**{k:v for k,v in arrays.items() if isinstance(v,np.ndarray)})
  rel=str(npz.relative_to(OUT));output_hashes[rel]=sha256_file(npz)
  rows=selected_rows[(selected_rows.page.astype(str)==page)&(selected_rows.candidate==candidate)]
  if len(rows)!=1:raise RuntimeError(f'{page} {candidate}: missing unique confirmed metric row')
  page_metric_rows.append(rows.iloc[0].to_dict())
  # Component diagnostics are crop-local if labels/stats are available.
  if isinstance(data,dict) and 'labels' in data and 'stats' in data and observed is not None:
   labels=data['labels'];stats=data['stats'];clean=data.get('clean')
   for cid in range(1,len(stats)):
    x,y,w,h,area=[int(v) for v in stats[cid,:5]]
    if area<20:continue
    comp_crop=(labels[y:y+h,x:x+w]==cid)
    rec_crop=recon[y:y+h,x:x+w]
    obs_crop=observed[y:y+h,x:x+w]
    changed=int(np.sum((rec_crop!=obs_crop)&comp_crop))
    row={'page':page,'candidate':candidate,'component':cid,'x':x,'y':y,'w':w,'h':h,'area':area,'changed_pixels':changed}
    if clean is not None:
     clean_crop=clean[y:y+h,x:x+w]
     row['component_mae']=float(np.abs(clean_crop.astype(np.float32)-rec_crop.astype(np.float32))[comp_crop].mean()) if comp_crop.any() else np.nan
    component_rows.append(row)
  del recon,arrays;gc.collect()
 pm=PAGE_RESULTS/f'{page}_metrics.csv';cm=PAGE_RESULTS/f'{page}_components.csv'
 atomic_csv(pd.DataFrame(page_metric_rows),pm);atomic_csv(pd.DataFrame(component_rows),cm)
 output_hashes[str(pm.relative_to(OUT))]=sha256_file(pm);output_hashes[str(cm.relative_to(OUT))]=sha256_file(cm)
 marker={'page':page,'candidates':SELECTED_CANDIDATES,'completed_utc':datetime.now(timezone.utc).isoformat(),'outputs':output_hashes}
 tmp=Path(str(PAGE_RESULTS/f'{page}_complete.json')+'.tmp');tmp.write_text(json.dumps(marker,indent=2));tmp.replace(PAGE_RESULTS/f'{page}_complete.json')
 print('Stage 4',index,'/',len(pages),page)
 del data,page_metric_rows,component_rows;gc.collect()

## Merge completed pages and create a compact diagnostic queue

This pass runs only after every page has a valid completion marker. It writes merged tables and a bounded list of pages/components to inspect. It does not create hundreds of figures during the measurement loop.

In [ ]:
incomplete=[p for p in pages if not valid_complete(p)]
if incomplete: raise RuntimeError(f'Stage 4 incomplete pages: {incomplete}')
page_frames=[pd.read_csv(PAGE_RESULTS/f'{p}_metrics.csv') for p in pages]
component_frames=[pd.read_csv(PAGE_RESULTS/f'{p}_components.csv') for p in pages if (PAGE_RESULTS/f'{p}_components.csv').stat().st_size>1]
page_df=pd.concat(page_frames,ignore_index=True);component_df=pd.concat(component_frames,ignore_index=True) if component_frames else pd.DataFrame()
page_df.to_csv(TABLES/'stage4_page_metrics.csv',index=False);component_df.to_csv(TABLES/'stage4_component_metrics.csv',index=False)
queue=[]
for candidate,g in page_df.groupby('candidate'):
 for label,col,asc in [('largest_false_preservation','added_false_annotation',False),('largest_true_print_gain','added_true_print',False),('lowest_precision','added_precision',True)]:
  for rank,(_,r) in enumerate(g.sort_values(col,ascending=asc).head(12).iterrows(),1):queue.append({'category':label,'rank':rank,'candidate':candidate,'page':r.page,'value':r[col]})
pd.DataFrame(queue).to_csv(TABLES/'diagnostic_review_queue.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.9.1','source_run':str(SOURCE_RUN),'original_notebook':str(ORIGINAL_NOTEBOOK),'selected_candidates':SELECTED_CANDIDATES,'pages':len(pages),'baseline_equivalence_passed':True,'all_page_confirmation_complete':True,'stage4_page_atomic_complete':True,'outside_mask_check':'enforced when engine exposes observed and mask','promotion_authorised':False}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('COMPLETE:',OUT)
display(summary);display(pd.DataFrame(queue).head(30))